# Safeguards before real-process qualification
## A guided team review • 12 November 2026

**DECISION: real-process qualification remains gated.** This notebook documents the proposed safeguards, the read-only environment findings and the smallest planned test batch. It does not implement or execute them.

| Ready to inspect | Proposed, not implemented here | Not established |
|---|---|---|
| Existing mock-tested candidate; source review; environment metadata | Independent guardian, process containment, bounded cleanup; ten-scenario plan | Real-process qualification, native Temporal qualification, measured results |

GitHub displays this notebook as a static document. **There are no code cells to run.** No Jupyter installation is needed to read it. Actual GitHub/Jupyter rendering has not been verified.

[Research-book contents](../../notebooks/00_start_here.ipynb) · [Study report](../../RESEARCH_STUDY.ipynb) · [Experimental candidate](README.md) · [Draft PR #7](https://github.com/node-and-norm/revocation-delegated-workflows/pull/7)


## Choose a reading route

**Organizational leaders:** read the decision, the safety architecture, the environment findings and the next gate.

**Researchers:** add the test matrix, evidence rules, verdicts and limits.

**Engineers and reviewers:** read every section, then inspect the pinned implementation and historical failure record. Do not treat this plan as an execution instruction or a passed control.

The central question is simple: **if the exchange gets stuck, can a separate mechanism stop the whole test and show what actually happened?**


## 1. Where we are now

Baseline: PR #7 at **`f1e50910d4cdd41010840520f209f1ae084138b1`**, inspected 12 November 2026. Its 37 mock/pure test methods are recorded as passing; they were **not rerun for this notebook**. No real child process, socket, Temporal Workflow/Activity or measured study ran in that candidate qualification.

The candidate persists dispatch intent, sends once, bounds a process join, rejects same-object reuse and preserves uncertainty. These are useful mechanisms, but their tests use process/channel/clock doubles.

**A bounded join is not a bounded whole operation.** Startup, filesystem persistence, termination and other OS operations can still block. The candidate is unintegrated development code, not a replacement for the native collector.

Sources: [pinned status](https://github.com/node-and-norm/revocation-delegated-workflows/blob/f1e50910d4cdd41010840520f209f1ae084138b1/experiments/temporal-qualification-v0.1/STATUS.md), [pinned exchange](https://github.com/node-and-norm/revocation-delegated-workflows/blob/f1e50910d4cdd41010840520f209f1ae084138b1/experiments/temporal-qualification-v0.1/bounded_exchange.py), [earlier partial-qualification decision](../../docs/decisions/2026-11-04-temporal-partial-qualification.md).


## 2. The proposed emergency-stop architecture

```text
Independent guardian (outside the test boundary)
    │ one launch instruction; fixed deadline; bounded status
    ▼
Dedicated, externally managed containment boundary
    ├── trusted launcher / controller
    ├── single-use exchange worker
    └── synthetic gateway / supervised evidence writer
```

1. Establish the boundary and its ownership **before** admitting an attempt.
2. Verify boundary READY before admission: guardian/boundary identity and configured limits. Then require per-attempt READY within the setup budget, binding attempt identity and descriptor custody. Neither is simply an unverified message.
3. After boundary READY, the guardian admits one attempt and starts its absolute deadline; exchange dispatch waits for per-attempt READY. It must not block in process startup, journal writes, hashing or pipe reads.
4. A supervised launcher performs potentially blocking setup. Capped, nonblocking status messages cannot reset the deadline.
5. At failure or timeout, use a separate lifecycle mechanism to stop the entire boundary. Never send a cleanup command through a possibly partial execute channel.
6. Independently observe exit of all in-scope tasks. A successful signal request is not proof they stopped.

**Preferred:** a documented delegated OS scope/cgroup or provider-managed disposable container with an external owner. Availability here is not established. A fresh coding container alone does not prove these properties.


## 3. Proposed timing and cleanup contract

These are **uncalibrated engineering budgets**, not study observation windows. Freeze them before any approved run; do not extend a live case to obtain a pass.

| Phase | Maximum / absolute rule |
|---|---|
| Guardian/boundary bootstrap | Before admission; no test dispatch permitted |
| Attempt setup and readiness | 10 seconds from admission |
| Single exchange | 5 seconds; end no later than second 15 |
| Graceful cleanup | 5 seconds, clipped to the absolute schedule |
| Forced containment and exit observation | Remaining budget; final cutoff at second 30 |

An earlier failure begins cleanup immediately. Heartbeats never renew the budget. Failure to prepare the guardian or boundary prevents admission.

At the graceful deadline, escalate through the boundary's force-stop mechanism. At the final cutoff, unknown process membership or unresolved termination means **cleanup_unresolved**. Stop the batch and do not reuse the transport or boundary. Retain ownership for operator/provider recovery.

On normal completion, capture evidence before cleanup. On emergency timeout, stop promptly even if capture is incomplete. Post-stop evidence must not fill an earlier observation gap.

This is not a hard-real-time guarantee against kernel hangs, host suspension or guardian failure. Those require a separately verified external service and fault model.


## 4. What the cloud inspection actually showed

Read-only inspection on **12 November 2026** used OS metadata and command/path discovery. No test workload, signal, socket, namespace operation or service-manager action was attempted.

| Observed metadata | What it supports | What it does not establish |
|---|---|---|
| Linux 6.18.57 x86_64; unprivileged user | Identity of the inspected runtime | Behavior of another/future environment |
| Capability masks all zero; NoNewPrivs=1; Seccomp=2 | Restricted current process context | Exact syscall policy or proof every operation is forbidden |
| `/proc/self/cgroup`: `0::/`; no visible cgroup mount; `/sys/fs/cgroup` absent | No exposed delegated cgroup control found | Host has no cgroups |
| PID 1: `codex`; ordinary systemd/D-Bus endpoints absent | No evidenced service-manager control route | All platform management is absent |
| `systemctl`, `systemd-run`, `timeout`, `nsenter`, `unshare` on PATH | Binaries are installed | Permission or verified containment semantics |
| Docker/Podman commands not found on PATH; usual sockets absent | No discovered local container-control route | No provider-managed containers exist |

**Conclusion:** the inspected environment does not expose the stronger controls needed by this design through the paths examined. Provider-level capabilities remain unknown, not disproved.

See [inspection evidence](safeguards/RUNTIME_INSPECTION.md) for exact commands and selected outputs. This is a transcribed record of observed tool output, not an independently signed host attestation or a runtime control test.

### Mac candidate: stopped status, not startup clearance

A separate read-only environment review reports that **both Colima and Lima list `node-norm-research` as stopped**. These are related management views, not independent proof of complete process absence or verified containment/stop semantics.

The saved profile includes **persistent disks, user-mode networking, host-name resolution, proxy propagation and broad automatic forwarding**. Existing workloads and restart policies have not been inspected. Starting the VM could resume retained services or workloads and expose network paths before a qualification fixture is ready; stopped status does not make startup inert.

A Rosetta-related `sysctl` read was denied and was **not retried**. No denial was bypassed. This chapter does not establish Rosetta/runtime suitability.

**Do not start this VM under the present documentation step.** Before a separate startup decision, review retained workload/restart behavior and the concrete network exposure, then establish the independent watchdog/containment controls. No security-setting change is implied. See the inspection record for attribution and limits.


## 5. What the critical review changed

| Tempting shortcut | Review finding | Required response |
|---|---|---|
| Kill a process group | Descendants can leave the group; scans race | Prefer a managed boundary; otherwise explicitly narrow to cooperative fixtures |
| Track a pidfd | Identifies one process, not a descendant tree | Do not infer group-wide identity or safe reuse of a numeric PGID |
| Start timer before `Popen` | Guardian can block during launch itself | Establish guardian/boundary first; move blocking launch inside supervision |
| Trust a stop acknowledgment | Signal delivery is not observed exit | Independently verify all in-scope tasks ended |
| Save all logs before stopping | Persistence can hang | Bound status and logging; never delay emergency containment for evidence |
| Weak-reference endpoint claim | Covers the same Python object only | Account for every descriptor duplicate; prohibit fresh-session retry |

The review was separately assigned **AI-assisted design review**, not independent human validation. Its conclusion was conditional design acceptance, not readiness to run.


## 6. Changes required before the real-process gate

- Add an independently supervised launcher and guardian with a documented failure model.
- Establish actual containment membership, ownership and a separately bounded stop path.
- Replace the historical controller's possibly blocking stop RPC and unbounded post-kill wait; do not silently inherit them into this harness.
- Record every intended endpoint duplicate and its lifetime. The worker may need a duplicate, so do not claim the guardian has sole physical ownership.
- Use fresh, unique attempt folders and permanently discard uncertain transports. No retry after possible send.
- Keep transport outcome, evidence completeness and cleanup result separate. `reply_observed` alone cannot qualify a case.
- Keep runtime exporters and the four-domain observation cutoff as later integration work. The existing `ObservationCut` trusts declared acknowledgments and does not prove an atomic snapshot.

These changes are **not implemented by this notebook**. [Observation-cut source](observation_cut.py) · [Candidate limitations](README.md).


## 7. Planned smallest real-process batch

**Not authorized to run merely by reading this notebook.** Preconditions: reviewed implementation, suitable permitted environment, verified safeguard capability and explicit execution approval. Inert synthetic fixtures only; no Temporal service, model/API calls or production actions.

| ID | Deliberate condition | Required observation / expected classification |
|---|---|---|
| T01 | Timely well-formed reply | Exactly one gateway receipt; bound reply; worker exit; complete records and cleanup |
| T02 | Peer never reads; send blocks | Actual blocked-send condition established; deadline containment; uncertain; no replay |
| T03 | Peer receives but never replies | Independent receipt retained; uncertain response; contained process |
| T04 | Reply frame starts but never completes | Partial-frame receipt established; readable is not complete; bounded containment |
| T05 | Peer closes during exchange | EOF/error retained; no retry; receipt evidence determines what is known |
| T06 | Valid reply clearly after cutoff | Never timely success; late evidence retained separately |
| T07 | Wrong request ID in saved result envelope, introduced by a reviewed fixture | No accepted transport success; misbound envelope preserved. This does not test a peer-supplied ID inside the opaque gateway payload. |
| T08 | Oversized reply | No accepted transport success; bounded handling and evidence |
| T09 | Intent-record write fails before send | Failure observed; zero gateway receipts within complete fixture observation; missing journal alone proves nothing |
| T10 | Child resists ordinary termination | External containment exercised; candidate cleanup failure remains visible even if outer recovery succeeds |

**Binding limitation:** the worker builds the outer result-envelope ID from the original request; the parent checks that envelope, while `gateway_result` is opaque. Peer-level response identity validation is not implemented by this check and needs a separate schema/design decision.

One attempt per declared scenario, maximum ten attempts. Fault induction must itself be reviewed and shown to have occurred; do not alter the production fixed request merely to force a pipe condition. If the current wire shape cannot safely exercise a proposed fault, revise that fixture plan before the batch rather than substitute an easier test and call it passed.

Deadline races, descriptor-alias enforcement, crash durability and hostile escape tests are outside this initial batch. Ten controls do not estimate operational reliability.


## 8. Evidence that must survive each attempt

**Identity:** source digest, environment/version, exact command/input, attempt ID and frozen budgets.

**Independent observations:** process/boundary identity, creation and exit observations, watchdog timing, gateway receipt count and complete synthetic request/response records.

**Candidate records:** dispatch intent, reply, worker error and supervisor record, including explicit missing/late status. Preserve primary and cleanup failures separately.

**Provenance:** immutable attempt directory, artifact digests and readable case table. Digests establish byte identity, not truthful or complete records.

There is no native history in this transport-only batch. Do not manufacture four-domain completeness. Termination is cleanup evidence, not proof that cancellation stopped an Activity or that no downstream effect occurred.


## 9. Pass, fail, uncertain — and when to stop

| Result | Meaning |
|---|---|
| PASS (control) | Intended fault occurred; expected conservative classification, bounded dispatch, containment and evidence requirements were met |
| FAIL (control) | Duplicate dispatch, false timely success, accepted misbound reply, tainted-channel reuse or exceeded containment contract |
| UNCERTAIN / NOT QUALIFIED | Fault not established, evidence missing/conflicting or process state unresolved |

A transport result of **uncertain** can be the correct result of a passing fault-control test. It is not evidence of no effect. Candidate cleanup and outer recovery must be reported separately.

**Stop the batch immediately** on permission denial, unexpected external access, duplicate dispatch, uncontrolled descendants, watchdog failure or unreliable observation. Preserve the attempt. No automatic retries or denominator removal. A corrective rerun needs a revised recorded plan.


## 10. Team responsibilities and AI disclosure

| Responsibility | Assignment / authority |
|---|---|
| Research direction and consequential approvals | Mark, human study lead |
| Coordination, source traceability and readable report | Alfred, AI-assisted coordinator |
| Implementation work, if separately approved | Assigned engineering role; source and test changes recorded |
| Critical review | Separately assigned reviewer; record AI assistance and unresolved findings |
| Environment ownership / recovery | Must be identified before execution; not presumed available |

SORA, MIRA and TRACE are part of the user's team vocabulary; this chapter does not invent particular assignments or imply they are persistent live agents. [Team charter](../../docs/team/CHARTER.md) supplies the role basis.

This chapter was assembled with AI assistance from source files, tool observations and separate AI-assisted critique. Design approval is not execution approval or empirical validation. It does not disclose complete private research conversations or claim such a transcript exists.

Jupyter provides the readable document format. The LangChain-inspired progression is **scope → evidence → review → synthesis**; no live LangGraph orchestration is added here.


## 11. The next decision and definition of done

```text
Current: design + metadata review
      ↓
Documented suitable containment capability
      ↓
Reviewed implementation + explicit execution approval
      ↓
Bounded real-process transport qualification report
      ↓
Separate native Temporal integration / qualification decision
      ↓
Separate measured-collection decision
```

**Today:** retain draft PR #7 as experimental. Do not run real-process qualification in the inspected runtime under a guaranteed-stop claim.

**This gate is complete only when** the exact candidate/environment has a reviewed transport-control report including unsuccessful and unresolved cases. That report is not native Temporal qualification, study completion, compliance certification or independent human validation.

If stronger containment cannot be established, bring an explicit weaker trusted-fixture proposal for review. Do not silently downgrade the safeguards.

[Back to candidate](README.md) · [Research-book contents](../../notebooks/00_start_here.ipynb) · [Study report](../../RESEARCH_STUDY.ipynb)
